# N025 · 相向双指针与有序配对

**目标：** 利用有序性安全排除不可能答案。

**先修：** N013, N017, N022。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 左右端点；单调排除；排序前提；回文过滤。

**配套讲解来源：** [同名逐章意见](../../comment/025_opposite_two_pointers.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两个"从两端往中间夹"的问题，共用一个思想：两个下标 left、right 分别站在序列的头和尾，每一步根据当前信息**有依据地扔掉一端**，逐步逼近答案。

**问题一（有序两数和，`two_sum_sorted`）**：给你一个**已经按升序排好**的数组和一个目标数 target，找出两个数使它们的和恰等于 target，返回这两个数的位置（1 基编号）。举个具体到数字的例子：输入 `[2,7,11,15]`、target=9 → 输出 `[1,2]` → 为什么？第 1 个数 2 加第 2 个数 7 恰好是 9。暴力做法要两两配对试 O(n²) 对，本章的做法靠"有序"这个前提，每一步安全淘汰一批不可能的候选，只扫一遍就出答案。

**问题二（回文判断，`is_palindrome`）**：给你一个字符串，只考虑其中的字母和数字字符、忽略大小写，问整个串是否前后对称。例如输入 `'A man, a plan, a canal: Panama'` → 输出 `True` → 为什么？把空格、逗号、冒号全部忽略并把大写统一成小写后，剩下 `amanaplanacanalpanama`，从两端向中间逐对对照全部相等；而 `'race a car'` 过滤后是 `raceacar`，对照到中间一对 e 对 a 不等，返回 False。两个指针从两端向中间走，天然适合"检查对称"这种两端配对的结构。

## 2. 基础知识：先读懂这些词

- **相向双指针（opposite-direction two pointers）**：一个指针从最左出发、一个从最右出发，两者相向而行、逐步收窄考察范围。它适合"配对、对称、夹逼"类问题，本章两个函数都是这个骨架。
- **左右端点（left / right）**：当前还"没被排除"区域的边界。任何时刻，可能成立的答案都落在这两个指针围成的范围之内；指针每动一步，范围缩一格。
- **单调排除（safe elimination）**：本章的灵魂。利用"数组有序"这一单调性做推理——当左值加右值已经小于 target 时，右端点固定不动、把左端点换成任何更靠右的数只会更大？不，换成更靠左已经不存在了；正确的推理是：左值与**剩下任何**候选（都不超过右值）配对的结果都不会超过当前这个和，全都小于 target，所以**左端点本人**已经不可能出现在任何答案里，可以放心扔掉。每扔一次就淘汰一整批候选，这正是 “正确性与复杂度”部分 说的"每次移动指针都排除一整行或一整列不可能解"。
- **排序前提（sortedness precondition）**：单调排除的推理链每一步都用到了"右边的数不比左边小"。数组无序时推理不成立，算法就可能错杀真正的答案——这是本章方法成立的硬前提（练习 2 专门让你构造反例体会）。
- **1 基索引（1-based index）**：位置从 1 数起而不是从 0 数起。LeetCode 167 题要求返回 1 基下标，所以代码里返回的是 `[left+1,right+1]`。
- **回文（palindrome）**：正读反读都一样的串。判断时只需检查"第 i 个字符等于倒数第 i 个字符"逐对成立。
- **回文过滤（filtering）**：判断前先约定"什么字符算数"。本章用 `isalnum()` 只保留字母和数字，用 `lower()` 把大小写拉平，空格和标点一概跳过。
- **isalnum / lower（字符工具）**：`c.isalnum()` 判断字符是否为字母或数字（Python 语义下对 Unicode 也生效）；`c.lower()` 把字符转成小写。两者配合完成"规范化大小写后逐对比较"。

## 3. 思路推导：从小例子开始

**`two_sum_sorted` 的思路：**

- **Step 1：两根指针就位。** left 指向最小值（下标 0），right 指向最大值（下标 n−1）。
- **Step 2：算当前和，三选一。** 若和等于 target，直接报出这对位置；若和太小，扔掉左端点（left 右移）；若和太大，扔掉右端点（right 左移）。
- **Step 3：重复到指针相遇。** 每一步要么出答案要么范围缩一格，最多 n 步内必然结束；相遇仍没找到就返回 None。

**为什么"和太小扔左边"是安全的？** 我们要确认被扔掉的左值确实不可能是答案的一半。当前和 = 左值 + 右值 < target，而右值是**当前范围内最大的**候选（数组升序）。也就是说，左值哪怕配上这个范围内最大的右值都凑不够，配上其他任何候选（都 ≤ 右值）只会更小，更凑不够。所以左值与范围内任何数的配对全部小于 target，左值无罪释放、永远出局是合理的。"和太大扔右边"是镜像推理：右值配上范围内最小的左值都超了，配别的只会更大。

**手算演示一：`two_sum_sorted([2,7,11,15], 9)`（对应 “自动测试”部分 第一条断言）**

| 轮 | left | right | 左值+右值 | 与 9 比较 | 动作 |
|---|---|---|---|---|---|
| 1 | 0 (2) | 3 (15) | 17 | 太大 | 扔右，right=2 |
| 2 | 0 (2) | 2 (11) | 13 | 太大 | 扔右，right=1 |
| 3 | 0 (2) | 1 (7) | 9 | 相等 | 返回 [0+1, 1+1] = [1,2] |

**手算演示二（“运行示例”部分 的表格重现）**：`a=[1,2,4,8,10]`、target=12，表格记录了每轮的指针与和：第一轮 left=0、right=4，1+10=11 太小，扔左；第二轮 left=1、right=4，2+10=12 命中，停。你看 “运行示例”部分 表格里的两行数据正是这两轮。

**`is_palindrome` 的思路：**

- **Step 1：两端指针就位。** left=0，right=末尾。
- **Step 2：先各自跳过"不算数"的字符。** 左指针向右跨过所有非字母数字字符，右指针向左同理。两个内层 while 干的就是这件事，跳完 left 和 right 都停在有效字符上。
- **Step 3：对照这一对。** 两个字符都转小写后比较：不等，立刻判负；相等，确认了一对对称字符，两端各进一格。
- **Step 4：指针相遇即成功。** 所有对称对都确认过了，返回 True。

**手算演示三：`is_palindrome('race a car')`**

| 轮 | 左字符 | 右字符 | 小写比较 | 结果 |
|---|---|---|---|---|
| 1 | r (下标0) | r (下标9) | 相等 | 各进一格 |
| 2 | a (1) | a (8) | 相等 | 各进一格 |
| 3 | c (2) | c (7) | 相等 | 各进一格 |
| 4 | e (3) | a (5) | **不等** | 返回 False |

注意两个指针都不会停在空格上：比如左指针从下标 3 的 e 之后若碰到空格会继续跳。而 `'A man, a plan, a canal: Panama'` 里每一对（A/a、m/m、a/a……）在忽略大小写后都相等，最终返回 True。

**手算演示四：`is_palindrome('A man, a plan, a canal: Panama')` 的前几对**

| 轮 | 左侧有效字符（下标） | 右侧有效字符（下标） | 小写比较 |
|---|---|---|---|
| 1 | A (0) | a (29) | a == a，通过 |
| 2 | m (2) | m (28) | m == m，通过 |
| 3 | a (3) | a (27) | a == a，通过 |
| … | （中间逐对对照，含 p 对 P 这类大小写配对） | | 全部通过，返回 True |

这串的总长是 30 个字符，过滤掉空格逗号冒号后剩 21 个有效字符，左右指针各走约 10 步就相遇，对照的成本只与有效字符数成正比。

## 4. 核心代码：two_sum_sorted

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def two_sum_sorted(numbers, target):
    left, right = (0, len(numbers) - 1)
    while left < right:
        total = numbers[left] + numbers[right]
        if total == target:
            return [left + 1, right + 1]
        if total < target:
            left += 1
        else:
            right -= 1
    return None
```

### 逐段读懂代码

### 函数 1：`two_sum_sorted`（有序两数和）

先贴 notebook “分段实现”部分 里的真实代码：

```python
def two_sum_sorted(numbers, target):
    left, right = (0, len(numbers) - 1)
    while left < right:
        total = numbers[left] + numbers[right]
        if total == target:
            return [left + 1, right + 1]
        if total < target:
            left += 1
        else:
            right -= 1
    return None
```

- **第 2 行 `left,right=0,len(numbers)-1`**：一行同时初始化两端指针——left 取 0（最小值处），right 取长度减一（最大值处）。left<right 保证不重复使用同一个元素。
- **第 3 行 `while left<right:`**：只要范围内还有至少两个元素就继续。指针相遇说明所有配对都被排除光了。
- **第 4 行 `total=numbers[left]+numbers[right]`**：缓存当前和，下面三个分支都要用它，避免重复相加。
- **第 5 行 `if total==target: return [left+1,right+1]`**：命中即返回。注意返回的是 **1 基**下标（各加一），这是 LeetCode 167 的题目要求；由于 left 始终小于 right，返回的两个编号也严格递增。
- **第 6 行 `if total<target: left+=1`**：和太小，按第三节的推理扔掉左端点——左值配上当前最大候选都凑不够，出局。
- **第 7 行 `else: right-=1`**：否则（和太大）镜像地扔掉右端点——右值配上当前最小候选都超了，出局。两个 else 分支合起来保证每轮范围必然缩小，循环必然终止。
- **第 8 行 `return None`**：指针相遇还没命中，说明不存在和为 target 的数对，用 None 明确表示"没找到"。测试里的对拍正是拿 `ans is not None` 与暴力解"是否存在有效对"互相对照的。

### 函数 2：`is_palindrome`（回文判断）

```python
def is_palindrome(s):
    left, right = (0, len(s) - 1)
    while left < right:
        while left < right and (not s[left].isalnum()):
            left += 1
        while left < right and (not s[right].isalnum()):
            right -= 1
        if s[left].lower() != s[right].lower():
            return False
        left += 1
        right -= 1
    return True
```

- **第 2 行**：与上一个函数完全相同的开局——两端指针就位。
- **第 4 行 `while left<right and not s[left].isalnum(): left+=1`**：跳过左侧不算数的字符。条件里 `left<right` 放在前面是双重保险：字符串全是标点（比如 `".,;"`）时左指针会被这个条件拦住，不会冲过右指针造成越界或自相矛盾的对照。短路求值保证越界判断永远先于取字符。
- **第 5 行**：右指针的镜像版本，向左跳过标点空白。
- **第 6 行 `if s[left].lower()!=s[right].lower(): return False`**：规范化后对照。`lower()` 把双方都拉到小写再比，于是 'A' 和 'a' 算相等。一旦不等，对称性被破坏，立即判负。
- **第 7 行 `left+=1; right-=1`**：这一对确认过了，左指针进一格、右指针退一格，两端同时向中间收拢，进入下一对。分号连写只是紧凑风格。
- **第 8 行 `return True`**：循环正常结束说明所有对称对都通过了对照。空串 `''` 直接跳过循环返回 True（空串视为回文），单字符串同样返回 True——这两个平凡情形不需要专门写分支，循环条件天然放过它们。

### 两个函数共享的骨架

你可以把两个函数并排看：都是"left、right 就位 → while left<right → 根据当前信息收窄 → 相遇即收场"。区别只在"收窄的依据"：两数和用大小关系推理出该扔哪边，回文用跳过和对照决定指针怎么动。掌握了这个骨架，相向双指针一族的问题都是它的变奏。

## 5. 分段实现：按顺序运行

**本章接口：** `two_sum_sorted(numbers, target)`、`is_palindrome(s)`

### two_sum_sorted

In [1]:
def two_sum_sorted(numbers, target):
    left, right = (0, len(numbers) - 1)
    while left < right:
        total = numbers[left] + numbers[right]
        if total == target:
            return [left + 1, right + 1]
        if total < target:
            left += 1
        else:
            right -= 1
    return None

### is_palindrome

In [2]:
def is_palindrome(s):
    left, right = (0, len(s) - 1)
    while left < right:
        while left < right and (not s[left].isalnum()):
            left += 1
        while left < right and (not s[right].isalnum()):
            right -= 1
        if s[left].lower() != s[right].lower():
            return False
        left += 1
        right -= 1
    return True

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,4,8,10]; l=0; r=len(a)-1; rows=[]
while l<r:
    total=a[l]+a[r]; rows.append((l,r,a[l],a[r],total))
    if total==12: break
    if total<12: l+=1
    else: r-=1
show_table(['left','right','左值','右值','和'],rows)

left,right,左值,右值,和
0,4,1,10,11
1,4,2,10,12


## 7. 正确性、适用条件与复杂度

每次移动指针都排除一整行或一整列不可能解；未排除解仍在当前指针范围内。回文每次确认对称一对字符，剩余中段是同一子问题。

**代价：** 两者 O(n) 时间、O(1) 辅助空间。字符分类采用Python语义；平台ASCII范围内与常见题意一致。

### 展开理解

**two_sum_sorted 为什么是对的？** 我们要说明两件事：它报出的答案是真答案；它没报答案时确实不存在答案。第一件事容易——返回前程序确实验证了 `numbers[left]+numbers[right]==target`，而且 left、right 是两个不同位置，答案货真价实。第二件事靠的是"扔得有理"：程序每扔掉一个端点，都 accompanied 着第三节的论证——被扔的那个值与**当时范围内所有**候选的配对和都清一色地小于（或大于）target，这些配对全都不是答案，扔掉它们不会误伤。换句说话，任何时刻"仍在范围内"的配对集合恰好包含着所有还没被证明不可能的配对。既然范围内的候选一个不漏地被考察（每次要么出答案要么缩小范围），那么若存在答案它必然在某个时刻被左右指针同时指到并返回；若循环走完没返回，说明所有配对都被逐一证明过不可能，确实无解。

**is_palindrome 为什么是对的？** 回文的定义是"过滤规范化后，第 i 个有效字符等于倒数第 i 个"。程序每轮先跳过无效字符（保证 left、right 都停在有效字符上），再对照一对并把两端收进一格——于是每完成一轮，就确认了一对最外侧的对称字符，剩下的中间部分是不是回文，是一个规模更小的同类问题。指针相遇时所有对都被确认，答案为真；任何一对不等则当场为假。跳过逻辑保证了空串、全标点串这些边界不会造成越界（内层 while 的 left<right 条件兜底）。

**复杂度是多少？** 两个函数都是 O(n) 时间、O(1) 额外空间。两数和的外层循环每轮至少让 left 或 right 前进一格，两个指针合起来最多走 n 步；回文的三个 while 加起来也只让每根指针单调走一遍。具体数字：n=10⁵ 的数组大约二十万次指针移动和加法，一瞬间完成；额外空间只有 left、right、total 三五个变量，与 n 无关。对比暴力两数和的 O(n²)：n=10⁵ 时约五十亿次配对，双指针快了五万倍。“正确性与复杂度”部分 还备注了字符分类按 Python 语义（isalnum 认 Unicode 字母数字），在题目限定 ASCII 的常见场景下与题意一致。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `assert two_sum_sorted([2,7,11,15],9)==[1,2]`：正常值，也是 LeetCode 167 的样例。测的是"命中返回 + 1 基编号"，同时验证了"和太大扔右"的推理链（如第三节手算演示一走了两步扔右才命中）。
- `assert is_palindrome('A man, a plan, a canal: Panama')`：正常成功值。测的是"跳过标点空格 + 大小写拉平"两套规范化叠加后仍能逐对吻合——标点、空格、大小写三重干扰同时在场。
- `assert is_palindrome('')`：空串边界。循环一次不进，直接 True，测的是最平凡输入不崩不误判。
- `assert not is_palindrome('race a car')`：失败值。过滤后是 raceacar，中间那对 e/a 撕破对称，测的是不等对能被当场抓住。
- **三重循环对拍**（含金量最高的一段）：`for n in range(6)` 枚举长度 0 到 5，`combinations_with_replacement([-1,0,1,2],n)` 枚举该长度下**所有升序数组**（从 4 个候选值中可重复选取，天然有序，正好满足算法前提），`for t in range(-2,5)` 枚举目标值。对每个组合做两层检查：第一，`assert (ans is not None)==bool(valid)`——算法说"找得到"当且仅当暴力枚举的所有数对里确实存在和为 t 的，测的是**无解时不误报、有解时不漏报**；第二，`if ans: assert (ans[0]-1,ans[1]-1) in valid`——把返回的 1 基编号减一回 0 基，确认它确实落在暴力解集合里，测的是**报出的答案本身合法**。这段对拍把小规模上的全部情况穷举验证了一遍，比单点样例可信得多。

全部通过后打印 `N025: 所有本章断言通过`。

In [4]:
assert two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]

assert is_palindrome('A man, a plan, a canal: Panama')

assert is_palindrome('') and (not is_palindrome('race a car'))

from itertools import combinations_with_replacement

for n in range(6):
    for a in combinations_with_replacement([-1, 0, 1, 2], n):
        for t in range(-2, 5):
            ans = two_sum_sorted(a, t)
            valid = [(i, j) for i in range(n) for j in range(i + 1, n) if a[i] + a[j] == t]
            assert (ans is not None) == bool(valid)
            if ans:
                assert (ans[0] - 1, ans[1] - 1) in valid

print('N025: 所有本章断言通过')

N025: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 返回0基与1基索引分别测试。

**练习 2：** 构造无序输入使模板失效。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（0 基与 1 基索引分别测试）**：方向是暴露"返回编号基准"这个隐性契约。提示：本章实现固定返回 1 基（`left+1`），你可以写一个返回 0 基的孪生版本，然后用同一批输入（如 `[2,7,11,15]`、target=9，期望分别 [0,1] 和 [1,2]）交叉测试，并混入"答案在数组中部"（如 `[1,2,4,8,10]`、target=12 → 0 基 [1,4] / 1 基 [2,5]）与"无解"（返回 None）的情形。验收时说清楚：如果调用方以为是 0 基，哪个测试会先报错？
- **练习 2（构造无序输入使模板失效）**：方向是亲手打破"排序前提"，体会单调排除为什么靠不住。提示：找一个升序解法会走过头的输入，比如 `[3,7,2,8]`、target=10——真正的答案是 2+8（下标 2 和 3），但无序数组上双指针的移动判断会失灵：left=0(3)、right=3(8)，和 11 太大扔右，8 被扔掉，真答案出局，最终返回 None。你要手算列出每轮指针位置和"扔掉谁"，指出哪一步的推理在无序数组上不再成立（"右值是最大候选"这个前提没了）。再补一个边界：数组只有两个元素、或答案恰是首尾两个元素时无序也会翻车吗？

两道练习做完，你应该能一句话说出本章方法成立的充要工作条件："数组必须先排好序，否则每次'安全排除'都不再安全。"

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def two_sum_sorted(numbers, target):
    left, right = (0, len(numbers) - 1)
    while left < right:
        total = numbers[left] + numbers[right]
        if total == target:
            return [left + 1, right + 1]
        if total < target:
            left += 1
        else:
            right -= 1
    return None

def is_palindrome(s):
    left, right = (0, len(s) - 1)
    while left < right:
        while left < right and (not s[left].isalnum()):
            left += 1
        while left < right and (not s[right].isalnum()):
            right -= 1
        if s[left].lower() != s[right].lower():
            return False
        left += 1
        right -= 1
    return True

# 示例与回归测试
assert two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]

assert is_palindrome('A man, a plan, a canal: Panama')

assert is_palindrome('') and (not is_palindrome('race a car'))

from itertools import combinations_with_replacement

for n in range(6):
    for a in combinations_with_replacement([-1, 0, 1, 2], n):
        for t in range(-2, 5):
            ans = two_sum_sorted(a, t)
            valid = [(i, j) for i in range(n) for j in range(i + 1, n) if a[i] + a[j] == t]
            assert (ans is not None) == bool(valid)
            if ans:
                assert (ans[0] - 1, ans[1] - 1) in valid

print('N025: 所有本章断言通过')

N025: 所有本章断言通过


## 11. 代表题与迁移

- **167. Two Sum II - Input Array Is Sorted（有序数组的两数之和）**：这题就是 `two_sum_sorted` 本尊，练的是单调排除推理和 1 基返回值的细节；原题数组已升序，正好省掉排序步骤、逼你用线性双指针而不是哈希或暴力。
- **125. Valid Palindrome（验证回文串）**：这题就是 `is_palindrome` 本尊，练的是"两端跳过无效字符 + 规范化大小写"的内层双 while 写法；全标点串、空串是它的经典边界。

两题都标为 canonical（与本章一一对应）。拓展方向：167 的思路搬到"三数之和"里做内层配对（N027 会见到），125 的两端对照换成"忽略若干种字符"的变体（如只比较字母）。自测问题：如果数组降序排列，`total<target` 时该扔哪边？（答案：扔右边，推理方向全部镜像。）

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。